# Inference - ensemble 
In this notebook we do inference using fine-tuned gemma-2 9b and llama-3 8b using T4 * 2 Gpu parallel, motivation behind to create this is the huge test size (25k samples). 

Prerequisite: Access to gemma-2 and llama-3

Please upvote if you find this helpful!

# Import libs

In [ ]:
!pip install -q --no-deps /kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2/transformers/transformers-4.42.0.dev0-py3-none-any.whl
!pip install -q --no-index --find-links /kaggle/input/notebooks/hengliu50605/gemma2-infer-wheels/wheels tokenizers==0.19.1 huggingface_hub==0.23.4 safetensors==0.4.3 peft==0.12.0 bitsandbytes==0.43.1

In [ ]:
import torch
import sklearn
import numpy as np
import pandas as pd
import time

import transformers
# from transformers import AutoTokenizer, Gemma2ForSequenceClassification, BitsAndBytesConfig
from transformers import AutoTokenizer, LlamaForSequenceClassification, BitsAndBytesConfig, Gemma2ForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType
from torch.cuda.amp import autocast
from threading import Thread

print('transformers', transformers.__version__)  # 應為 4.42.0.dev0

torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

if (not torch.cuda.is_available()): print("Sorry - GPU required!")


In [ ]:
import json, os, time
import numpy as np, pandas as pd, torch

COMP_DIR = '/kaggle/input/competitions/llm-classification-finetuning'
TARGETS = ['winner_model_a', 'winner_model_b', 'winner_tie']

ENSEMBLE_W = 0.75   # gemma 的權重;llama 得到 1 - 0.70。
                    # 驗證集上 0.65~0.75 差不到 0.0002,取整數比較不會過擬合。

MAX_TOKENS = 6144
MAX_BATCH = 32      # 再加一道筆數上限,避免極短序列組出過大的批次

SIX = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj']

CFG_GEMMA = dict(
    tag='gemma',
    arch='gemma2',
    model_name='/kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2',
    weights='/kaggle/input/notebooks/henryliu513436/gemma-2-9b-fine-tuning/model_gemma_2_run1_33k_len1536_cp_1.pth',
    target_modules=SIX,          # Run 1 用六模組
    max_length=1536,
    max_tokens=MAX_TOKENS, max_batch=MAX_BATCH,
    lora_r=16, lora_alpha=32,
    use_tta=True,
    prompt_max=256, safety=8,
    test_parquet='/kaggle/working/test_clean.parquet',
)

CFG_LLAMA = dict(
    tag='llama',
    arch='llama',
    model_name='/kaggle/input/llama-3/transformers/8b-chat-hf/1',   # 有 config.json 的那一層
    weights='/kaggle/input/notebooks/hengliu50605/lmsys-llama-3-tpu-train/model_gemma_2_run1_33k_len1536_cp_1.pth',
    target_modules=SIX,         # ← 去 llama 訓練 log 搜 trainable params 確認:
                                 #   約 13.6M = 四模組 / 約 32.5M = 六模組(改成 SIX)
    max_length=1536,             # ← 必須與 llama 訓練時的 CFG.MAX_LENGTH 相同
    max_tokens=MAX_TOKENS, max_batch=MAX_BATCH,
    lora_r=16, lora_alpha=32,
    use_tta=False,                # 時間不夠時可改 False,省一次前向(見最後一節)
    prompt_max=256, safety=8,
    test_parquet='/kaggle/working/test_clean.parquet',
)

for c in (CFG_GEMMA, CFG_LLAMA):
    with open(f"cfg_{c['tag']}.json", 'w') as f:
        json.dump(c, f, ensure_ascii=False, indent=1)
    assert os.path.exists(c['weights']), f"找不到權重: {c['weights']}"
    assert os.path.exists(c['model_name']), f"找不到基礎模型: {c['model_name']}"
print('設定已寫出,路徑都存在')

# Preprocessing

In [ ]:
def process(input_str):
    """與訓練時完全相同"""
    stripped_str = input_str.strip('[]')
    sentences = [s.strip('"') for s in stripped_str.split('","')]
    return ' '.join(sentences)

test = pd.read_csv(f'{COMP_DIR}/test.csv')
sample_sub = pd.read_csv(f'{COMP_DIR}/sample_submission.csv')
for col in ['prompt', 'response_a', 'response_b']:
    test[col] = test[col].apply(process)

test[['id', 'prompt', 'response_a', 'response_b']].to_parquet(
    '/kaggle/working/test_clean.parquet', index=False)

print('test:', len(test), '| sample_submission:', len(sample_sub))
assert len(test) == len(sample_sub)
assert set(test['id']) == set(sample_sub['id']), 'test 與 sample_submission 的 id 集合不同'
test.head(2)

# inference script

In [ ]:
%%writefile predict.py
"""單一模型的推論腳本。由 ensemble notebook 以子行程呼叫:

    python predict.py cfg_gemma.json

跑在獨立行程裡的用意:行程結束時作業系統會【完整】收回它佔用的 GPU 記憶體。
在 notebook 內用 del + empty_cache() 常常收不乾淨(快取配置器、記憶體碎片、
Out[] 歷史殘留的參照),兩個 9B 級模型會因此塞不下 2xT4。

輸出 prob_<tag>.npy:形狀 (len(test), 3),已做完 TTA 平均,欄位順序與原始 test 一致。
"""
import json
import sys
import time
from threading import Thread

import numpy as np
import pandas as pd
import torch
from torch.cuda.amp import autocast
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType

# Gemma-2 的 logit soft-capping 需要完整的 attention 分數矩陣,融合核心做不到
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

CFG = json.load(open(sys.argv[1]))
TAG = CFG['tag']
MAX_LENGTH = CFG['max_length']
MAX_TOKENS = CFG['max_tokens']   # 每批的 padded token 上限(取代固定 batch size)
MAX_BATCH = CFG['max_batch']     # 再加一道筆數上限,避免極短序列組出超大批次
USE_TTA = CFG['use_tta']
PROMPT_MAX = CFG['prompt_max']
SAFETY = CFG['safety']

assert torch.cuda.device_count() >= 2, f'需要 2 張 GPU,目前只有 {torch.cuda.device_count()}'
print(f'=== {TAG} === max_length={MAX_LENGTH} max_tokens={MAX_TOKENS} tta={USE_TTA}', flush=True)

test = pd.read_parquet(CFG['test_parquet'])
print('samples:', len(test), flush=True)

# ---- tokenizer:必須與該模型訓練時完全相同 ----
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
tokenizer.add_eos_token = True

# ---- 組文字 + 分段截斷(與訓練 notebook 一字不差) ----
TEMPLATE = ('User prompt: ', '\n\nModel A :\n', '\n\n--------\n\nModel B:\n')


def build_text(prompt, resp_first, resp_second):
    return TEMPLATE[0] + prompt + TEMPLATE[1] + resp_first + TEMPLATE[2] + resp_second


def allocate(len_a, len_b, budget):
    """A、B 平分 budget;短的一方用不完的額度讓給另一方"""
    half = budget // 2
    if len_a <= half:
        return len_a, min(len_b, budget - len_a)
    if len_b <= half:
        return min(len_a, budget - len_b), len_b
    return half, budget - half


def build_texts_truncated(prompts, first, second):
    enc = lambda s: tokenizer(list(s), add_special_tokens=False)['input_ids']
    overhead = len(tokenizer(''.join(TEMPLATE), add_special_tokens=False)['input_ids']) + 2
    budget = MAX_LENGTH - overhead - SAFETY
    texts, needs_trunc = [], []
    for p_txt, a_txt, b_txt, p, a, b in zip(prompts, first, second,
                                            enc(prompts), enc(first), enc(second)):
        if len(p) + len(a) + len(b) <= budget:
            texts.append(build_text(p_txt, a_txt, b_txt))
            needs_trunc.append(False)
            continue
        p = p[:PROMPT_MAX]
        na, nb = allocate(len(a), len(b), budget - len(p))
        texts.append(build_text(tokenizer.decode(p),
                                tokenizer.decode(a[:na]),
                                tokenizer.decode(b[:nb])))
        needs_trunc.append(True)
    return texts, np.array(needs_trunc)


texts, needs_trunc = build_texts_truncated(test['prompt'], test['response_a'], test['response_b'])
if USE_TTA:
    texts_swap, _ = build_texts_truncated(test['prompt'], test['response_b'], test['response_a'])
    texts += texts_swap

encoded = tokenizer(texts, truncation=True, max_length=MAX_LENGTH)['input_ids']
lengths = np.array([len(ids) for ids in encoded])
print(f'forward passes: {len(texts)} | token mean {lengths.mean():.0f} max {lengths.max()}', flush=True)
print(f'原本會超過長度的樣本: {needs_trunc.mean():.1%} | '
      f'仍被硬砍到 {MAX_LENGTH} 的: {np.mean(lengths == MAX_LENGTH):.2%}', flush=True)

# ---- 載入模型 ----
if CFG['arch'] == 'gemma2':
    from transformers import Gemma2ForSequenceClassification as ModelCls
    extra = dict(attn_implementation='eager')   # soft-capping 必需
elif CFG['arch'] == 'llama':
    from transformers import LlamaForSequenceClassification as ModelCls
    extra = {}
else:
    raise ValueError(f"未知的 arch: {CFG['arch']}")

bnb_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.float16)


def load_base_model(device_map):
    # T4 不支援 bfloat16 -> 用 float16
    model = ModelCls.from_pretrained(
        CFG['model_name'],
        num_labels=3,
        torch_dtype=torch.float16,
        quantization_config=bnb_config,
        device_map=device_map,
        **extra)
    model.config.pad_token_id = tokenizer.pad_token_id
    return model


peft_config = LoraConfig(
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=0.05,
    bias='none',
    inference_mode=True,
    task_type=TaskType.SEQ_CLS,
    target_modules=CFG['target_modules'])

state_dict = torch.load(CFG['weights'], map_location='cpu')
print('keys in .pth:', len(state_dict), flush=True)
assert any('score' in k for k in state_dict), '.pth 裡找不到分類頭,訓練端存錯了'


def attach_weights(base_model):
    model = get_peft_model(base_model, peft_config)
    result = model.load_state_dict(state_dict, strict=False)
    if result.unexpected_keys:
        raise ValueError(f'權重載不進去,名稱不符(target_modules 與訓練時不同?): '
                         f'{result.unexpected_keys[:5]}')
    # PEFT 把 score 包成 original_module(凍結,用不到)與 modules_to_save.default(真正在用的)。
    # 訓練端只存 requires_grad 的參數,所以 original_module 缺席是正常的。
    bad = [k for k in result.missing_keys if 'score' in k and 'original_module' not in k]
    if bad:
        raise ValueError(f'分類頭沒載到,推論結果會是隨機的: {bad}')
    return model.eval()


device0, device1 = torch.device('cuda:0'), torch.device('cuda:1')
model_0 = attach_weights(load_base_model('cuda:0'))
model_1 = attach_weights(load_base_model('cuda:1'))
for i in range(2):
    print(f'cuda:{i} 已使用 {torch.cuda.memory_allocated(i)/1e9:.1f} GB', flush=True)


# ---- 推論 ----
def make_batches(indices):
    """indices 已由長到短排序。用固定 token 預算分批,而非固定筆數。

    一批 pad 後的 token 數 = 筆數 x 批內最長。固定 batch size 的問題是它必須
    遷就最長的序列,於是短序列的批次只用掉一小部分 GPU 容量。改成預算制之後
    長序列自動用小批、短序列用大批,每批都餵滿。

    這只改變樣本的【分組方式】,每一筆的預測結果不變(padding 位置由
    attention_mask 遮蔽,分類頭取的位置也只取決於序列本身)。
    """
    batches, cur = [], []
    for i in indices:
        longest = len(encoded[cur[0]]) if cur else len(encoded[i])   # 降冪排序 -> 首筆最長
        if cur and ((len(cur) + 1) * longest > MAX_TOKENS or len(cur) >= MAX_BATCH):
            batches.append(cur)
            cur = [i]
        else:
            cur.append(i)
    if cur:
        batches.append(cur)
    return batches


def inference(indices, model, device, results, worker_id):
    out = {}
    batches = make_batches(indices)
    t_start = time.time()
    for bi, batch_idx in enumerate(batches):
        max_len = max(len(encoded[i]) for i in batch_idx)
        input_ids = torch.full((len(batch_idx), max_len), tokenizer.pad_token_id, dtype=torch.long)
        attention_mask = torch.zeros((len(batch_idx), max_len), dtype=torch.long)
        for row, i in enumerate(batch_idx):
            ids = encoded[i]
            input_ids[row, :len(ids)] = torch.tensor(ids)
            attention_mask[row, :len(ids)] = 1
        with torch.no_grad(), autocast():
            logits = model(input_ids=input_ids.to(device),
                           attention_mask=attention_mask.to(device)).logits
        probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        for row, i in enumerate(batch_idx):
            out[i] = probs[row]

        # 進度與 ETA:9 小時上限下,要能【提早】判斷跑不跑得完
        if (bi + 1) % 100 == 0 or bi + 1 == len(batches):
            done, total = bi + 1, len(batches)
            el = time.time() - t_start
            eta = el / done * (total - done)
            print(f'  [worker {worker_id}] {done}/{total} 批 | 已用 {el/60:.1f} 分 '
                  f'| 預估剩 {eta/60:.1f} 分', flush=True)
    torch.cuda.empty_cache()
    results[worker_id] = out


st = time.time()
# 由長到短排序:長度相近的進同一批 -> pad 最少;交錯分給兩張卡讓工作量平均
order = np.argsort(-lengths).tolist()
results = {}
t0 = Thread(target=inference, args=(order[0::2], model_0, device0, results, 0))
t1 = Thread(target=inference, args=(order[1::2], model_1, device1, results, 1))
t0.start(); t1.start()
t0.join(); t1.join()

all_probs = {**results[0], **results[1]}
probs = np.stack([all_probs[i] for i in range(len(texts))])   # 還原成原始順序
print(f'{TAG} 推論完成,耗時 {time.time() - st:.1f}s', flush=True)

n = len(test)
p_orig = probs[:n]
if USE_TTA:
    # 對調版本的「A贏」其實是原本的 B 贏,先換回來再平均
    p_final = (p_orig + probs[n:][:, [1, 0, 2]]) / 2
else:
    p_final = p_orig

assert p_final.shape == (n, 3), f'形狀不對: {p_final.shape}'
assert np.isfinite(p_final).all(), '出現 NaN/Inf'
np.save(f'prob_{TAG}.npy', p_final.astype(np.float32))
print(f'已存 prob_{TAG}.npy  三類平均 {p_final.mean(axis=0).round(4)}', flush=True)


# Run Gemma

In [ ]:
!python predict.py cfg_gemma.json

# Run Llama

In [ ]:
!python predict.py cfg_llama.json

# Ensemble

In [ ]:
pg = np.load('prob_gemma.npy')
pl = np.load('prob_llama.npy')
assert pg.shape == pl.shape == (len(test), 3), f'形狀不對: {pg.shape} {pl.shape}'

p = ENSEMBLE_W * pg + (1 - ENSEMBLE_W) * pl
p = p / p.sum(axis=1, keepdims=True)

pred = pd.DataFrame({'id': test['id'],
                     'winner_model_a': p[:, 0],
                     'winner_model_b': p[:, 1],
                     'winner_tie':     p[:, 2]})
# 用 id 對齊,不依賴 test 與 sample_submission 的列順序相同
sub = sample_sub[['id']].merge(pred, on='id', how='left')

assert sub[TARGETS].isna().sum().sum() == 0, 'id 對不起來,有缺值'
assert np.isfinite(sub[TARGETS].values).all(), '有 NaN/Inf'
assert np.allclose(sub[TARGETS].sum(axis=1), 1, atol=1e-6), '機率沒有加總為 1'
sub.to_csv('submission.csv', index=False)

print('已寫出 submission.csv  筆數', len(sub))
print('三類平均:', sub[TARGETS].mean().round(4).to_dict(), ' (訓練集約 0.35/0.34/0.31)')
print('gemma 與 llama 預測的平均絕對差:', np.abs(pg - pl).mean().round(4),
      ' (太接近代表集成沒有多樣性)')
sub.head()


Inference completes in ~8 hrs, there are still stuff to improve upon this. I would encourage to try out different post-processing and share. Kaggle way :) 